# Öncü Bilgi Adayı Taraması — Ders Kitabı Notebooku

Bu notebook, Model 11 sonrasındaki **Seçenek 2** yönünü denetlenebilir bir masa başı araştırmasına dönüştürür. Amaç performans üretmek değil; mevcut feature haritasındaki gerçek boşlukları kapatabilecek veri ailelerini, veri çekmeden önce as-of ve erişim kapılarından geçirmektir.

Sabitler: `noter_devir_otomobil_adet`, `down/stable/up`, ±%5 stable bandı, haftalık güncellenen cari-ay nowcast, iki aylık bilgi disiplini ve kilitli test.

## Neden yeni bilgi ailesi?

Model 11, yalnız mevcut bilgi temsilleri altında saptanabilir beceri bulamadı. Bu sonuç yeni bir algoritma aramayı değil, ölçülmemiş bir mekanizmayı aramayı gerektirir. Yeni aday; kur, faiz oranı, takvim veya hedef gecikmesinin yeni bir dönüşümü olmamalı, gerçek bir davranışsal ya da piyasa-içi boşluğu kapatmalıdır.

In [1]:
import pandas as pd

adaylar = pd.DataFrame([
    {
        'aday': 'BDDK haftalık taşıt kredisi bakiyesi/değişimi',
        'bilgi_ailesi': 'Finansmana fiili erişim/kullanım',
        'ham_frekans': 'haftalık',
        'ilk_tarih': 'Ocak 2014 (kesin ilk hafta doğrulanmadı)',
        'M_eksi_2': 'takvim düzeyinde evet',
        'vintaj_riski': 'orta-yüksek',
        'on_hukum': 'koşullu devam',
    },
    {
        'aday': 'BETAM-sahibindex ilan arzı ve ilan yaşı',
        'bilgi_ailesi': 'İlan arzı, stok ve ilan yaşı',
        'ham_frekans': 'aylık rapor',
        'ilk_tarih': 'Aralık 2023 ilk yayın; üç yıl backfill',
        'M_eksi_2': 'cari kullanımda muhtemelen evet',
        'vintaj_riski': 'yüksek',
        'on_hukum': 'mevcut protokol için elendi',
    },
    {
        'aday': 'Google Trends araç-alım arama ilgisi',
        'bilgi_ailesi': 'Gerçek zamanlı işlem niyeti',
        'ham_frekans': 'günlük/haftalık',
        'ilk_tarih': 'UI 2004; tutarlı alpha API 1.800 gün',
        'M_eksi_2': 'cari kullanımda evet',
        'vintaj_riski': 'çok yüksek',
        'on_hukum': 'geriye dönük test için elendi',
    },
])
adaylar

,aday,bilgi_ailesi,ham_frekans,ilk_tarih,M_eksi_2,vintaj_riski,on_hukum
0,BDDK haftalık taşıt kredisi bakiyesi/değişimi,Finansmana fiili erişim/kullanım,haftalık,Ocak 2014 (kesin ilk hafta doğrulanmadı),takvim düzeyinde evet,orta-yüksek,koşullu devam
1,BETAM-sahibindex ilan arzı ve ilan yaşı,"İlan arzı, stok ve ilan yaşı",aylık rapor,Aralık 2023 ilk yayın; üç yıl backfill,cari kullanımda muhtemelen evet,yüksek,mevcut protokol için elendi
2,Google Trends araç-alım arama ilgisi,Gerçek zamanlı işlem niyeti,günlük/haftalık,UI 2004; tutarlı alpha API 1.800 gün,cari kullanımda evet,çok yüksek,geriye dönük test için elendi


## Kapıların mekanik denetimi

Bu denetim performans seçmez. Yalnız kapsamın Pusula tarafından konan 1–3 aday sınırına uyduğunu, her adayın ayrı bir gerçek boşluğu kapattığını ve yalnız bir adayın sonraki erişim fizibilitesine taşındığını doğrular.

In [2]:
assert 1 <= len(adaylar) <= 3
assert adaylar['bilgi_ailesi'].nunique() == len(adaylar)
assert (adaylar['on_hukum'] == 'koşullu devam').sum() == 1
assert not adaylar.isna().any().any()
adaylar.groupby('on_hukum', as_index=False).size()

,on_hukum,size
0,geriye dönük test için elendi,1
1,koşullu devam,1
2,mevcut protokol için elendi,1


## Aday 1 — BDDK taşıt kredisi

BDDK haftalık bülteni tüketici kredileri içinde taşıt kalemini yayımlar; metaveri haftalık serinin Ocak 2014'e uzandığını söyler. Bu aday mevcut taşıt kredisi **faizinden** farklıdır: fiyatı değil, gerçekleşmiş kredi bakiyesini ölçer. Yine de başvuru/onay değildir ve net bakiye hem yeni kullandırım hem geri ödeme taşır.

Kritik as-of sorunu revizyondur. BDDK geçmiş dönem değerlerinin takip eden yayınlarda değişebileceğini açıklar. Bu nedenle güncel seri geçmiş originlere doğrudan bağlanamaz; tarihli ilk-yayım bültenlerinin korunumu bir sonraki küçük aşamada kanıtlanmalıdır.

Kaynaklar: [Haftalık Bülten](https://www.bddk.org.tr/Veri/Detay/158), [gelişmiş gösterim](https://www.bddk.org.tr/BultenHaftalik/tr/Gelismis), [metaveri](https://www.bddk.org.tr/BultenDosyalari/Home/Index/Haftalik-MetaVeri), [yayın takvimi](https://www.bddk.org.tr/Veri/Detay/71), [SSS/revizyon](https://www.bddk.org.tr/Sss/Liste/110).

## Aday 2 — BETAM–sahibindex

İlan sayısı, kapatılan ilan, satılan/satılık oranı ve ilanda kalma süresi hedef mekanizmasına yakındır. Ancak ilk kamuya açık aylık yayın Aralık 2023'tür. İlk rapor geçmiş üç yılı geriye dönük ele alır; bu backfill, 2020–2023 originlerinde gerçekten bilinen ilk-yayım değerlerini oluşturmaz. Dolayısıyla ekonomik uygunluk, as-of uygunluğa dönüşmez.

Kaynaklar: [Aralık 2023 ilk rapor](https://betam.bahcesehir.edu.tr/2023/12/sahibindex-otomobil-piyasasi-gorunumu/), [Temmuz 2026 güncel örnek](https://betam.bahcesehir.edu.tr/2026/07/sahibindex-otomobil-piyasasi-gorunumu-temmuz-2026/), [rapor arşivi](https://betam.bahcesehir.edu.tr/kategori/ekonomik-arastirmalar/yayinlar/otomotiv-piyasasi-gorunumu/).

## Aday 3 — Google Trends

Arama ilgisi işlem öncesi niyet için cazip bir vekildir. Buna karşılık Trends UI örnekleme, istatistiksel gürültü ve sorgu penceresine göre 0–100 yeniden ölçekleme uygular. Tutarlı ölçekli API yaklaşık beş yıllık kayan pencere sunar ve sınırlı alpha erişimindedir. Sabit bir geçmiş vintaj arşivi olmadan bugünkü sorguyla geçmiş originlerin o günkü bilgisini yeniden kurduğumuzu iddia edemeyiz.

Kaynaklar: [Google Trends FAQ](https://support.google.com/trends/answer/4365533?hl=en), [Google Trends API alpha](https://developers.google.com/search/blog/2025/07/trends-api).

## Karar ve sonraki kapı

Yalnız BDDK adayı **erişim/vintaj fizibilitesine** taşınır; bu bir model veya feature terfisi değildir. Sonraki aşama kullanıcı onayı gerektirir ve yalnız kesin yayın gecikmesi, ilk-yayım arşivi, revizyon davranışı, erişim yolu ve beş örnek tarihte vintaj karşılaştırması üretmelidir. As-of kapısı geçmeden veri seti/model hattı kurulmaz; geçse bile önce oracle `null95 + 0,15` tavan testi uygulanır.